In [ ]:
import os
import pandas as pd
import pybedtools
import matplotlib.pyplot as plt

In [ ]:
base_dir = "/Path/to/Project/"

prdm9_13mer_dir = os.path.join(base_dir, "final_analysis/data/prdm9/prdm9b_Human_1_to_6.bed")
prdm9_B_dir = os.path.join(base_dir, "final_analysis/data/prdm9/prdm9b_Human7.bed")


# ------------------------
# Load motifs
# ------------------------
fimo_cols = [
    "chrom", "Start", "End", "motif_id", 
    "strand", "score", "pvalue", "qvalue", "motif_seq"
]

# 13-mer PRDM9 A motif
prdm13mer = pd.read_csv(
    prdm9_13mer_dir,
    sep="\t",
    comment="#",
    header=None,
    names=fimo_cols
)[["chrom", "Start", "End"]]
prdm13mer = prdm13mer[~prdm13mer["chrom"].isin(["chrX", "chrY", "chrM"])]

# PRDM9 B-specific motif (Human7)
prdmB = pd.read_csv(
    prdm9_B_dir,
    sep="\t",
    comment="#",
    header=None,
    names=fimo_cols
)[["chrom", "Start", "End"]]
prdmB = prdmB[~prdmB["chrom"].isin(["chrX", "chrY", "chrM"])]


motifs = {"PRDM-13mer": prdm13mer, "PRDM-B": prdmB}


print(len(prdmB))
print(len(prdm13mer))

In [ ]:
pan_available = os.path.join(base_dir, "CHM13v2.telo_cent.complement.bed")
pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region

In [ ]:
def filter_prdm_to_allowed_regions(
    prdm_df: pd.DataFrame,
    allowed_df: pd.DataFrame,
    chrom_col_prdm: str = "chrom",
    start_col_prdm: str = "Start",
    end_col_prdm: str = "End",
    chrom_col_allowed: str = "Chrom",
    start_col_allowed: str = "Start",
    end_col_allowed: str = "End",) -> pd.DataFrame:
    """
    Keep PRDMb records that overlap any allowed region on the same chromosome.
    No clipping, no splitting.
    """

    out = []

    # process chromosome by chromosome (fast + clean)
    for chrom, pr_chr in prdm_df.groupby(chrom_col_prdm):
        allowed_chr = allowed_df[
            allowed_df[chrom_col_allowed] == chrom
        ][[start_col_allowed, end_col_allowed]]

        if allowed_chr.empty:
            continue

        allowed = allowed_chr.sort_values(
            [start_col_allowed, end_col_allowed]
        ).to_numpy()

        pr = pr_chr[[start_col_prdm, end_col_prdm]].to_numpy()

        j = 0
        for s, e in pr:
            # advance allowed pointer
            while j < len(allowed) and allowed[j][1] <= s:
                j += 1

            k = j
            keep = False
            while k < len(allowed) and allowed[k][0] < e:
                if e > allowed[k][0] and s < allowed[k][1]:
                    keep = True
                    break
                k += 1

            if keep:
                out.append((chrom, s, e))

    if not out:
        return prdm_df.iloc[0:0].copy()

    df_out = pd.DataFrame(out, columns=[chrom_col_prdm,
                                        start_col_prdm,
                                        end_col_prdm])
    # merge back any extra columns if needed
    df_out = df_out.merge(
        prdm_df,
        on=[chrom_col_prdm, start_col_prdm, end_col_prdm],
        how="left"
    )

    return df_out.reset_index(drop=True)


In [ ]:
prdmB = filter_prdm_to_allowed_regions(prdmB, pan_available_region)
prdm13mer = filter_prdm_to_allowed_regions(prdm13mer, pan_available_region)

print(len(prdmB))
print(len(prdm13mer))

In [ ]:
# Population directories
cha_pan = os.path.join(base_dir, "final_analysis/data/CHA/pan/bp35w60")
cha_ngs = os.path.join(base_dir, "final_analysis/data/CHA/ngs/bp35w60")
chb_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHB")
chs_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHS")
jpt_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/JPT")
khv_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/KHV")
cdx_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CDX")
fin_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/FIN")
ceu_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CEU")
yri_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/YRI")


datasets = {
    "CHA_pan": cha_pan,
    "CHA_ngs": cha_ngs,
    "CHB": chb_unmasked_dir,
    "CHS": chs_unmasked_dir,
    "JPT": jpt_unmasked_dir,
    "KHV": khv_unmasked_dir,
    "CDX": cdx_unmasked_dir,
    "FIN": fin_unmasked_dir,
    "CEU": ceu_unmasked_dir,
    "YRI": yri_unmasked_dir,
}

chroms = [str(i) for i in range(1, 23)]

# ------------------------
# File patterns
# ------------------------
file_patterns = {
    cha_pan: "CHA_recombmap_chr{chrom}_bp35w60",
    cha_ngs: "CHA_recombmap_chr{chrom}_bp35w60",
    chb_unmasked_dir: "CHB_chr{chrom}_no_mask.txt",
    chs_unmasked_dir: "CHS_chr{chrom}_no_mask.txt",
    jpt_unmasked_dir: "JPT_chr{chrom}_no_mask.txt",
    khv_unmasked_dir: "KHV_chr{chrom}_no_mask.txt",
    cdx_unmasked_dir: "CDX_chr{chrom}_no_mask.txt",
    fin_unmasked_dir: "FIN_chr{chrom}_no_mask.txt",
    ceu_unmasked_dir: "CEU_chr{chrom}_no_mask.txt",
    yri_unmasked_dir: "YRI_chr{chrom}_no_mask.txt",
}


def load_map(directory, chrom):
    if directory not in file_patterns:
        raise ValueError(f"Unknown directory: {directory}")
    file_name = file_patterns[directory].format(chrom=chrom)
    file_path = os.path.join(directory, file_name)
    
    if directory in [cha_pan, cha_ngs]:
        df = pd.read_csv(file_path, sep="\t", header=None, names=["Start", "End", "Rec.Rate"])
    else:
        df = pd.read_csv(file_path)
    return df

In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:

    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    # Two-pointer sweep (fast enough; map windows are usually not huge)
    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out



In [ ]:
def merge_neighboring_windows(df, chrom_col="chrom", start_col="Start", end_col="End"):
    """
    Merge adjacent intervals within each chromosome when previous end == next start.
    Assumes df has columns: chrom, Start, End.

    Returns merged dataframe with same columns.
    """
    if df.empty:
        return df.copy()

    df = df[[chrom_col, start_col, end_col]].copy()
    df = df.sort_values([chrom_col, start_col, end_col]).reset_index(drop=True)

    merged_rows = []

    for chrom, g in df.groupby(chrom_col, sort=False):
        starts = g[start_col].to_numpy()
        ends = g[end_col].to_numpy()

        cur_s = starts[0]
        cur_e = ends[0]

        for s, e in zip(starts[1:], ends[1:]):
            # merge rule: "previous end == next start or previous end + 1 == next start"
            if cur_e == s or cur_e + 1 == s:
                cur_e = e
            else:
                merged_rows.append((chrom, cur_s, cur_e))
                cur_s, cur_e = s, e

        merged_rows.append((chrom, cur_s, cur_e))

    merged = pd.DataFrame(merged_rows, columns=[chrom_col, start_col, end_col])
    return merged

In [ ]:
hotspot_dir = os.path.join(base_dir, "final_analysis/hotspot")

for dataset_name, dataset_dir in datasets.items():
    print(f"Processing dataset: {dataset_name}")
    total_genome_bp = 0
    
    all_hotspots = []
    for chrom in chroms:
        df = load_map(dataset_dir, chrom)
        df = clip_map_to_allowed_regions(df_map=df,
                                        allowed_df=pan_available_region,
                                        chrom=str(chrom),
                                        chrom_col_allowed="chr",
                                        start_col_allowed="Start",
                                        end_col_allowed="End")
        window_lens = df["End"] - df["Start"]
        total_genome_bp += window_lens.sum()
        avg_rate = (window_lens * df["Rec.Rate"]).sum() / window_lens.sum()
        hotspots = df[df["Rec.Rate"] > 10 * avg_rate].copy()
        hotspots["chrom"] = f"chr{chrom}"
        hotspots["Start"] = hotspots["Start"].round().astype(int)
        hotspots["End"] = hotspots["End"].round().astype(int)
        all_hotspots.append(hotspots[["chrom", "Start", "End"]])
    if not all_hotspots:
        continue
    
    hotspot_bed = pd.concat(all_hotspots, ignore_index=True)
    
    hotspot_bed = merge_neighboring_windows(
        hotspot_bed,
        chrom_col="chrom",
        start_col="Start",
        end_col="End"
    )
    hotspot_file = os.path.join(hotspot_dir, f"{dataset_name}_unmask_hotspots_Mar9.bed")
    # uncomment the line below to save the hotspot bed file
    hotspot_bed.to_csv(hotspot_file, sep="\t", header=False, index=False)
    



In [ ]:
def add_chr_prefix(df, col):
    df = df.copy()
    df[col] = df[col].astype(str)
    df[col] = df[col].apply(lambda x: x if x.startswith("chr") else f"chr{x}")
    return df

# Apply to all relevant dataframes
pan_available_region = add_chr_prefix(pan_available_region, "chr")
prdmB = add_chr_prefix(prdmB, "chrom")
prdm13mer = add_chr_prefix(prdm13mer, "chrom")
#hotspot_bed = add_chr_prefix(hotspot_bed, "chrom")

In [ ]:
import pybedtools
import numpy as np
import pandas as pd

accessible_df = pan_available_region[["chr", "Start", "End"]].copy()
accessible_bt = pybedtools.BedTool.from_dataframe(accessible_df).sort()

motifs_bt = {
    "PRDM-B": pybedtools.BedTool.from_dataframe(
        prdmB[["chrom", "Start", "End"]]
    ).sort(),

    "PRDM-13mer": pybedtools.BedTool.from_dataframe(
        prdm13mer[["chrom", "Start", "End"]]
    ).sort()
}


BLOCK_SIZE = 100_000

def make_blocks(accessible_df, block_size=100_000):
    blocks = []

    for _, row in accessible_df.iterrows():
        chrom, start, end = row["chr"], row["Start"], row["End"]
        for b in range(start, end, block_size):
            blocks.append([
                chrom,
                b,
                min(b + block_size, end)
            ])

    return pd.DataFrame(
        blocks, columns=["chrom", "block_start", "block_end"]
    )

	

In [ ]:
def assign_blocks(prdm_df, block_df):
    prdm_bt = pybedtools.BedTool.from_dataframe(
        prdm_df[["chrom", "Start", "End"]]
    )
    block_bt = pybedtools.BedTool.from_dataframe(
        block_df[["chrom", "block_start", "block_end"]]
    )

    return prdm_bt.intersect(block_bt, wa=True, wb=True).to_dataframe(
        names=[
            "chrom", "Start", "End",
            "block_chrom", "block_start", "block_end"
        ]
    )


In [ ]:
def shuffle_by_block_full(prdm_block_df, block_df, seed=None):
    rng = np.random.default_rng(seed)
    shuffled = []

    # Keep only the first block per PRDM site; to avoid duplicate counting when a prdm site overlap with multiple blocks
    prdm_block_df = prdm_block_df.sort_values(
        ["chrom", "Start", "block_start"]
    ).drop_duplicates(subset=["chrom", "Start", "End"], keep="first")

    for chrom in block_df["chrom"].unique():
        blocks_chr = block_df[block_df["chrom"] == chrom].reset_index(drop=True)
        prdm_chr = prdm_block_df[prdm_block_df["chrom"] == chrom]

        perm = rng.permutation(len(blocks_chr))

        for i, src_block in blocks_chr.iterrows():
            tgt_block = blocks_chr.iloc[perm[i]]
            offset = tgt_block["block_start"] - src_block["block_start"]

            moved = prdm_chr[
                prdm_chr["block_start"] == src_block["block_start"]
            ].copy()

            if moved.empty:
                continue

            moved["Start"] += offset
            moved["End"] += offset
            shuffled.append(moved[["chrom", "Start", "End"]])

    return pd.concat(shuffled, ignore_index=True)


In [ ]:
def plot_and_save_shuffle_histogram(
    hotspot_file,
    prdm_df,
    pan_available_region,
    population_name,
    motif_name,
    out_dir,
    n_perm=1000,
    block_size=100000
):
    os.makedirs(out_dir, exist_ok=True)

    # BedTools objects
    hotspot_bt = pybedtools.BedTool(hotspot_file)
    prdm_bt = pybedtools.BedTool.from_dataframe(
        prdm_df[["chrom", "Start", "End"]]
    ).sort()

    # Observed statistic
    observed = hotspot_bt.intersect(prdm_bt, u=True).count()

    # Build blocks and assign PRDM9 sites
    block_df = make_blocks(
        pan_available_region[["chr", "Start", "End"]],
        block_size
    )
    prdm_block_df = assign_blocks(prdm_df, block_df)

    # Null distribution
    null = np.zeros(n_perm, dtype=int)

    for i in range(n_perm):
        print(f"Permutation {i+1}/{n_perm} for population {population_name}, motif {motif_name}")
        shuffled_df = shuffle_by_block_full(
            prdm_block_df, block_df, seed=i
        )
        shuffled_bt = pybedtools.BedTool.from_dataframe(
            shuffled_df
        ).sort()

        null[i] = hotspot_bt.intersect(
            shuffled_bt, u=True
        ).count()

    # -----------------------
    # Save null distribution
    # -----------------------
    null_df = pd.DataFrame({
        "perm_id": np.arange(1, n_perm + 1),
        "overlap_count": null
    })

    null_file = os.path.join(
        out_dir,
        f"{population_name}_{motif_name}_null_overlap_counts_Mar9.tsv"
    )
    null_df.to_csv(null_file, sep="\t", index=False)

    # -----------------------
    # Plot histogram
    # -----------------------
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.hist(null, bins=100)
    plt.axvline(
        observed,
        color="red",
        linewidth=2,
        label=f"Observed = {observed}"
    )
    plt.xlabel("# Hotspot sites overlapping ≥1 PRDM9 site")
    plt.ylabel("Frequency")
    plt.title(f"{population_name} – {motif_name}")
    plt.legend()
    plt.tight_layout()

    plot_file = os.path.join(
        out_dir,
        f"{population_name}_{motif_name}_shuffle_histogram_Mar9.png"
    )
    plt.savefig(plot_file, dpi=300)
    plt.show()

    return {
        "observed": observed,
        "null_mean": null.mean(),
        "null_file": null_file,
        "plot_file": plot_file
    }



In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

# Function wrapper with all arguments as a dict
def run_enrichment(args):
    return plot_and_save_shuffle_histogram(**args)

In [ ]:
hotspot_dir = os.path.join(base_dir, "final_analysis/hotspot")
tasks = []

for dataset_name in datasets:
    hotspot_file = os.path.join(
        hotspot_dir, f"{dataset_name}_unmask_hotspots_Mar9.bed"
    )

    for motif_name, motif_df in motifs.items():
        task_args = {
            "hotspot_file": hotspot_file,
            "prdm_df": motif_df,
            "pan_available_region": pan_available_region,
            "population_name": dataset_name,
            "motif_name": motif_name,
            "out_dir": hotspot_dir,   
            "n_perm": 1000
        }
        tasks.append(task_args)


In [ ]:

print(f"Using up to {min(8, os.cpu_count())} threads")
max_threads = min(8, os.cpu_count())  # e.g., 8 threads or number of cores

results = []

with ThreadPoolExecutor(max_workers=max_threads) as executor:
    # submit all tasks
    future_to_task = {executor.submit(run_enrichment, t): t for t in tasks}

    # as they complete, collect results
    for future in as_completed(future_to_task):
        task = future_to_task[future]
        try:
            res = future.result()
            print(f"{task['population_name']} – {task['motif_name']} done")
            results.append(res)
        except Exception as e:
            print(f"Error in {task['population_name']} – {task['motif_name']}: {e}")
